In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb

In [ ]:
data = pd.read_csv(r'C:\Users\HP\Downloads\Telegram Desktop\german_credit_data.csv')

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
cols = ["Saving accounts", "Checking account"]
data[cols]=data[cols].fillna(data.mode().iloc[0])

In [ ]:
data.describe(include='all')

In [ ]:
sb.histplot(data['Age'],kde=True)

In [ ]:
sb.histplot(data['Credit amount'],kde=True)

In [ ]:
sb.countplot(x=data['Risk'])

In [ ]:
data['Risk'].value_counts()

In [ ]:
X= data.drop(['Risk'],axis=1)
y=data['Risk']

In [ ]:
obj_list = list(X.select_dtypes(include=['object']).columns)
obj_list

In [ ]:
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
ct =  ColumnTransformer([('oe',OrdinalEncoder(),obj_list)],remainder='passthrough')

In [ ]:
X=ct.fit_transform(X)

In [ ]:
X

In [ ]:
le=LabelEncoder()

In [ ]:
y=le.fit_transform(y)

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    stratify=y, 
                                                    test_size=0.2,random_state=10)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
sm = SMOTE(random_state = 10)
X1_res, y1_res = sm.fit_resample(X_train,y_train)

In [ ]:
X1_res = np.array(X1_res)
y1_res = np.array(y1_res)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X1_res,y1_res,test_size=0.2, random_state=10)

In [ ]:
X_train.shape, X_val.shape, y_train.shape, y_val.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
#Creating a dictionary of all models
model_dict = {}

model_dict['Logistic regression']= LogisticRegression(solver='liblinear',random_state=10)
model_dict['Decision Tree Classifier'] = DecisionTreeClassifier(random_state=10)
model_dict['Random Forest Classifier'] = RandomForestClassifier(random_state=10)

In [ ]:
#function to print accuracy of all models
def model_test(X_train, X_test, y_train, y_test,model,model_name):
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test) 
    accuracy = accuracy_score(y_test,y_pred)
    print('======================================{}======================================='.format(model_name))
    print('Score is : {}'.format(accuracy))
    
    print()

In [ ]:
for model_name,model in model_dict.items():
    model_test(X_train, X_val, y_train, y_val, model, model_name)

In [ ]:
for model_name, model in model_dict.items():
    p = model.predict(X_test)
    print('Testing accuracy of ',model_name,'=',accuracy_score(y_test,p))

In [ ]:
rfc=RandomForestClassifier(random_state=10)
rfc.fit(X_train,y_train)

In [ ]:
pred_val=rfc.predict(X_val)
pred_test=rfc.predict(X_test)


In [ ]:
cm_val = confusion_matrix(y_val,pred_val)
cm_val

In [ ]:
cm_test=confusion_matrix(y_test,pred_test)
cm_test

In [ ]:
#Plotting confusion matrix
cf_matrix = confusion_matrix(y_val, pred_val)
sb.heatmap(cf_matrix/np.sum(cf_matrix), annot=True, fmt='.2%')
plt.title('Confusion Matrix of Random Forest Classifier')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [ ]:
cols=list(X.columns)

In [ ]:
d=[[67,'male',2,'own','little','little',1169,6,'radio/TV']]
d1=[[22,'female',2,'own','little','moderate',5951,48,'radio/TV']]
d=pd.DataFrame(d,columns=cols)
d.head()


In [ ]:
pred=rfc.predict(ct.transform(d))
pred

In [ ]:
import pickle,joblib
pickle.dump(rfc,open('model.pkl','wb'))
joblib.dump(ct,'ct')